# PDNC disease NER backend

This notebook loads the three published models (SapBERT tagger, SmolLM tagger, projection head) on a Colab GPU and serves them to the web app.

**How to run it**

1. Runtime > Change runtime type > **T4 GPU**.
2. Runtime > **Run all**. Loading takes a few minutes the first time.
3. Copy the backend URL printed under "Start the server" and paste it into the web app's connection box.
4. Keep this tab open while you demo. If Colab disconnects, run all again; the URL will change, so paste the new one.

## Settings

In [ ]:
# ---- settings: the only cell you should need to edit ----
TUNNEL       = "cloudflare"   # "cloudflare" needs no account. If your network blocks it, use "ngrok".
NGROK_TOKEN  = ""             # only for TUNNEL = "ngrok": free token from https://dashboard.ngrok.com/get-started/your-authtoken
FRONTEND_URL = ""             # optional: your Vercel URL, e.g. "https://pdnc-ner.vercel.app" - prints a ready-to-open link

## Install

In [ ]:
!pip -q install -U transformers huggingface_hub safetensors fastapi "uvicorn[standard]" python-multipart
if TUNNEL == "ngrok":
    !pip -q install pyngrok

## Helpers
Same span, metric and bootstrap code as `all_test.ipynb`.

In [ ]:
import os, io, re, sys, time, ast, json, uuid, queue, hashlib, threading, traceback, subprocess, importlib.util
import numpy as np, pandas as pd, requests
from sklearn.metrics import f1_score, accuracy_score

SMOLLM_REPO  = "Aswath220/biomed_ner_smollm"
SAPBERT_REPO = "Aswath220/sapBERT_NER_finetune"
HEAD_REPO    = "Aswath220/sapbert-disease-projection-head"
NCBI_URL     = "https://raw.githubusercontent.com/spyysalo/ncbi-disease/master/conll/test.tsv"

# veto thresholds tuned on NCBI validation (same values as all_test.ipynb)
DROP_THR = {"smollm": 0.526, "sapbert": 0.4}
MODEL_NAMES = {"smollm": "SmolLM", "sapbert": "SapBERT"}
LABELS = ("O", "B-Disease", "I-Disease")

def spans_of(tags, typ="Disease"):
    out, st = [], None
    for i, x in enumerate(list(tags) + ["O"]):
        if st is not None and x != "I-" + typ: out.append((st, i)); st = None
        if x == "B-" + typ or (x == "I-" + typ and st is None): st = i
    return out

def tags_from_spans(n, spans):
    t = ["O"] * n
    for a, b in spans: t[a] = "B-Disease"; t[a + 1:b] = ["I-Disease"] * (b - a - 1)
    return t

def to_list(x):
    x = str(x).strip(); return ast.literal_eval(x) if x.startswith("[") else x.split()

def norm_tag(t): return LABELS[int(t)] if str(t).isdigit() else str(t)

def evaluate(gold, pred):
    G = {(i, a, b) for i, t in enumerate(gold) for a, b in spans_of(t)}
    P = {(i, a, b) for i, t in enumerate(pred) for a, b in spans_of(t)}
    tp = len(G & P); p = tp / max(len(P), 1); r = tp / max(len(G), 1)
    g, q = [x for s in gold for x in s], [x for s in pred for x in s]
    return {"span P": p, "span R": r, "span F1": 2 * p * r / (p + r) if p + r else 0.0,
            "token macro-F1": f1_score(g, q, labels=list(LABELS), average="macro", zero_division=0),
            "token acc": accuracy_score(g, q)}

def bootstrap(gold, base, withhead, n=2000, seed=0):
    """paired bootstrap over sentences: how often does the head version beat the base version?"""
    cnt = lambda pred: np.array([(len(set(spans_of(g)) & set(spans_of(p))), len(spans_of(p)), len(spans_of(g)))
                                 for g, p in zip(gold, pred)])
    cb, ch = cnt(base), cnt(withhead)
    f1 = lambda c: 2 * c[0] / max(c[1] + c[2], 1)
    rng = np.random.default_rng(seed)
    d = np.array([f1(ch[i].sum(0)) - f1(cb[i].sum(0)) for i in (rng.integers(0, len(cb), len(cb)) for _ in range(n))])
    return f1(ch.sum(0)) - f1(cb.sum(0)), np.percentile(d, 2.5), np.percentile(d, 97.5), (d > 0).mean()

def parse_output(text):
    text = text.split("\n")[0].strip()
    return [] if text.lower() in ("", "none") else [e.strip() for e in text.split("|") if e.strip()]

def ents_to_tags(words, ents):
    tags, used, low = ["O"] * len(words), [False] * len(words), [w.lower() for w in words]
    for e in sorted(set(ents), key=lambda e: -len(e.split())):
        ew = e.split(); L = len(ew)
        hits = [i for i in range(len(words) - L + 1) if words[i:i + L] == ew] or \
               [i for i in range(len(words) - L + 1) if low[i:i + L] == [x.lower() for x in ew]]
        for i in hits:
            if any(used[i:i + L]): continue
            tags[i] = "B-Disease"; tags[i + 1:i + L] = ["I-Disease"] * (L - 1); used[i:i + L] = [True] * L
    return tags

def fix_bio(tags):
    out = list(tags)
    for i, t in enumerate(out):
        if t == "I-Disease" and (i == 0 or out[i - 1] == "O"): out[i] = "B-Disease"
    return out

def veto(tags, scores, thr):
    """drop spans the head scores below thr (spans the head could not score are kept)"""
    keep = [sp for sp in spans_of(tags) if not (scores.get(sp, 1.0) < thr)]
    return tags_from_spans(len(tags), keep)

print("helpers ready")

## Load models

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForTokenClassification
from huggingface_hub import hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Go to Runtime > Change runtime type > T4 GPU, then Run all again.")
device    = "cuda"
AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
GPU_NAME  = torch.cuda.get_device_name(0)

# --- SmolLM generative tagger ---
llm_tok = AutoTokenizer.from_pretrained(SMOLLM_REPO)
if llm_tok.pad_token is None: llm_tok.pad_token = llm_tok.eos_token
llm_tok.padding_side = "left"
llm = AutoModelForCausalLM.from_pretrained(SMOLLM_REPO, dtype=AMP_DTYPE).to(device).eval()
try:
    PROMPT = json.load(open(hf_hub_download(SMOLLM_REPO, "train_info.json")))["prompt"]
except Exception:
    PROMPT = ("Find all disease mentions in the sentence. Copy them exactly, separated by ' | ', or write 'none'.\n"
              "Sentence: {}\nDiseases:")
print("SmolLM loaded")

# --- fine-tuned SapBERT token classifier ---
tag_tok = AutoTokenizer.from_pretrained(SAPBERT_REPO)
tagger  = AutoModelForTokenClassification.from_pretrained(SAPBERT_REPO, dtype=torch.float32).to(device).eval()
ID2LABEL = {int(k): v for k, v in tagger.config.id2label.items()}
assert set(ID2LABEL.values()) == set(LABELS), ID2LABEL
print("SapBERT tagger loaded")

# --- projection head (uses projection_head.py published in its repo) ---
_path = hf_hub_download(HEAD_REPO, "projection_head.py")
_spec = importlib.util.spec_from_file_location("projection_head", _path)
ph = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(ph)
scorer = ph.DiseaseScorer.from_pretrained(HEAD_REPO, device=device)
print("Projection head loaded")

# ---------- predictors: cb(done, total) reports progress to the web app ----------
@torch.no_grad()
def smollm_predict(sents, cb=lambda d, t: None, bs=32):
    out = []
    for i in range(0, len(sents), bs):
        chunk = sents[i:i + bs]
        enc = llm_tok([PROMPT.format(" ".join(w)) for w in chunk], return_tensors="pt", padding=True,
                      add_special_tokens=False).to(device)
        gen = llm.generate(**enc, max_new_tokens=96, do_sample=False,
                           pad_token_id=llm_tok.pad_token_id, eos_token_id=llm_tok.eos_token_id)
        texts = llm_tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        out += [ents_to_tags(w, parse_output(t)) for w, t in zip(chunk, texts)]
        cb(min(i + bs, len(sents)), len(sents))
    return out

@torch.no_grad()
def sapbert_predict(sents, cb=lambda d, t: None, bs=64):
    out = []
    for i in range(0, len(sents), bs):
        chunk = sents[i:i + bs]
        enc = tag_tok(chunk, is_split_into_words=True, truncation=True, max_length=512,
                      padding=True, return_tensors="pt").to(device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            ids = tagger(**enc).logits.argmax(-1).cpu().numpy()
        for b, words in enumerate(chunk):
            tags, seen = ["O"] * len(words), set()
            for j, wid in enumerate(enc.word_ids(b)):
                if wid is not None and wid not in seen: seen.add(wid); tags[wid] = ID2LABEL[int(ids[b, j])]
            out.append(fix_bio(tags))
        cb(min(i + bs, len(sents)), len(sents))
    return out

def head_span_scores(sents, tag_list, cb=lambda d, t: None):
    """one SapBERT pass per sentence, scoring every span the tagger proposed"""
    out = []
    for i, w in enumerate(sents):
        spans = sorted(set(spans_of(tag_list[i])))
        out.append(dict(zip(spans, scorer.score_spans(w, spans))) if spans else {})
        if i % 20 == 0 or i + 1 == len(sents): cb(i + 1, len(sents))
    return out

PREDICTORS = {"smollm": smollm_predict, "sapbert": sapbert_predict}
print(f"All models ready on {GPU_NAME}")

## API
Runs one job at a time on the GPU. Each run tags the file once, scores the spans with the head, and returns both the *without head* and *with head* results, so the head toggle in the app is instant. Results are cached, so re-running the same file and model is immediate. The `/tag` endpoint tags free text typed into the app.

In [ ]:
from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response

# ---------------- datasets ----------------
DATASETS = {}   # key -> {"name", "ids", "tokens", "gold" (or None)}

def load_ncbi():
    data, w, t = [], [], []
    for line in requests.get(NCBI_URL, timeout=120).text.splitlines():
        if not line.strip():
            if w: data.append((w, t)); w, t = [], []
            continue
        a, b = line.split("\t")[:2]; w.append(a); t.append(b)
    if w: data.append((w, t))
    DATASETS["ncbi"] = {"name": "NCBI-disease test", "ids": list(range(len(data))),
                        "tokens": [x[0] for x in data], "gold": [x[1] for x in data]}

def parse_upload(raw, filename):
    try:
        df = pd.read_csv(io.BytesIO(raw))
    except Exception as e:
        raise HTTPException(400, f"Could not read {filename} as CSV: {e}")
    if "tokens" in df.columns:
        toks = [[str(x) for x in to_list(v)] for v in df["tokens"]]
    elif "text" in df.columns:
        toks = [str(v).split() for v in df["text"]]
    else:
        raise HTTPException(400, f"{filename} needs a 'tokens' or 'text' column. Found: {', '.join(df.columns)}")
    if not toks:
        raise HTTPException(400, f"{filename} has no rows.")
    ids = df["id"].tolist() if "id" in df.columns else list(range(len(df)))
    gold = None
    if "ner_tags" in df.columns:
        gold = [[norm_tag(x) for x in to_list(v)] for v in df["ner_tags"]]
        bad = [i for i, (g, w) in enumerate(zip(gold, toks)) if len(g) != len(w)]
        if bad:
            raise HTTPException(400, f"ner_tags and tokens have different lengths in {len(bad)} rows "
                                     f"(first: row {bad[0]}). Fix the file or remove the ner_tags column.")
        unknown = {t for g in gold for t in g} - set(LABELS)
        if unknown:
            raise HTTPException(400, f"Unknown labels in ner_tags: {', '.join(sorted(unknown))}. Expected O, B-Disease, I-Disease or 0/1/2.")
    return {"name": filename, "ids": ids, "tokens": toks, "gold": gold}

# ---------------- jobs ----------------
JOBS, CACHE, PREDS = {}, {}, {}      # CACHE/PREDS keyed by (model, dataset_key)
job_q = queue.Queue()
GPU_LOCK = threading.Lock()          # file runs and free-text tagging never use the GPU at the same time

def build_result(ds, base, head, model, seconds):
    gold, toks = ds["gold"], ds["tokens"]
    res = {"model": model, "model_name": MODEL_NAMES[model], "threshold": DROP_THR[model],
           "dataset": {"name": ds["name"], "sentences": len(toks), "labelled": gold is not None},
           "seconds": round(seconds, 1), "variants": {}}
    for v, pred in (("base", base), ("head", head)):
        res["variants"][v] = {"spans": sum(len(spans_of(p)) for p in pred),
                              "metrics": {k: round(float(x), 4) for k, x in evaluate(gold, pred).items()} if gold else None}
    if gold:
        d, lo, hi, win = bootstrap(gold, base, head)
        res["bootstrap"] = {"delta_f1": round(float(d), 4), "ci_low": round(float(lo), 4),
                            "ci_high": round(float(hi), 4), "head_better_pct": round(float(win) * 100, 1)}
    changed = [i for i, (a, b) in enumerate(zip(base, head)) if a != b]
    ex = lambda i: {"index": i, "id": str(ds["ids"][i]), "tokens": toks[i], "base": base[i], "head": head[i],
                    "gold": gold[i] if gold else None}
    res["changed_count"] = len(changed)
    res["changed"] = [ex(i) for i in changed[:60]]
    res["preview"] = [ex(i) for i in range(min(25, len(toks)))]
    return res

def run_job(jid):
    j = JOBS[jid]; model, key = j["model"], j["dataset"]; ds = DATASETS[key]
    if (model, key) in CACHE:
        j.update(status="done", result=CACHE[(model, key)]); return
    j["status"] = "running"; t0 = time.time()
    def stage(label):
        def cb(done, total): j["progress"] = {"stage": label, "done": done, "total": total}
        cb(0, len(ds["tokens"])); return cb
    try:
        with GPU_LOCK:
            base = PREDICTORS[model](ds["tokens"], stage(f"Tagging with {MODEL_NAMES[model]}"))
            scores = head_span_scores(ds["tokens"], base, stage("Checking spans with the projection head"))
        head = [veto(t, s, DROP_THR[model]) for t, s in zip(base, scores)]
        j["progress"] = {"stage": "Scoring", "done": 1, "total": 1}
        res = build_result(ds, base, head, model, time.time() - t0)
        CACHE[(model, key)], PREDS[(model, key)] = res, {"base": base, "head": head}
        j.update(status="done", result=res)
        print(f"[job {jid}] {MODEL_NAMES[model]} on {ds['name']} done in {res['seconds']}s")
    except Exception as e:
        traceback.print_exc(); j.update(status="error", error=f"{type(e).__name__}: {e}")

def worker():                       # one job at a time so the GPU never runs two models at once
    while True:
        jid = job_q.get()
        try: run_job(jid)
        finally: job_q.task_done()
threading.Thread(target=worker, daemon=True).start()

# ---------------- API ----------------
app = FastAPI(title="PDNC disease NER backend")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"],
                   expose_headers=["Content-Disposition"])

@app.get("/health")
def health():
    return {"ok": True, "gpu": GPU_NAME, "models": list(MODEL_NAMES), "thresholds": DROP_THR,
            "ncbi_sentences": len(DATASETS.get("ncbi", {}).get("tokens", [])), "queued": job_q.qsize()}

@app.post("/jobs")
async def create_job(model: str = Form(...), dataset: str = Form("upload"), file: UploadFile | None = File(None)):
    if model not in MODEL_NAMES: raise HTTPException(400, f"Unknown model '{model}'. Use smollm or sapbert.")
    if dataset == "ncbi":
        if "ncbi" not in DATASETS: raise HTTPException(503, "NCBI test set is not loaded on the server.")
        key = "ncbi"
    else:
        if file is None: raise HTTPException(400, "Attach a CSV file or choose the NCBI test set.")
        raw = await file.read()
        key = "up_" + hashlib.sha1(raw).hexdigest()[:12]
        if key not in DATASETS: DATASETS[key] = parse_upload(raw, file.filename or "upload.csv")
    jid = uuid.uuid4().hex[:10]
    JOBS[jid] = {"id": jid, "model": model, "dataset": key, "status": "queued", "progress": None,
                 "result": None, "error": None}
    if (model, key) in CACHE: run_job(jid)          # already computed: answer instantly
    else: job_q.put(jid)
    return {"job_id": jid, "status": JOBS[jid]["status"]}

@app.get("/jobs/{jid}")
def get_job(jid: str):
    if jid not in JOBS: raise HTTPException(404, "Unknown job. The Colab runtime may have restarted; run it again.")
    j = JOBS[jid]
    return {k: j[k] for k in ("id", "model", "status", "progress", "result", "error")}

@app.get("/jobs/{jid}/download")
def download(jid: str, variant: str = "head"):
    j = JOBS.get(jid)
    if not j or j["status"] != "done": raise HTTPException(404, "No finished job with that id.")
    if variant not in ("base", "head"): raise HTTPException(400, "variant must be base or head")
    ds, pred = DATASETS[j["dataset"]], PREDS[(j["model"], j["dataset"])][variant]
    csv = pd.DataFrame({"id": ds["ids"], "ner_tags": [" ".join(p) for p in pred]}).to_csv(index=False)
    fname = f"submission_{j['model']}{'_head' if variant == 'head' else ''}.csv"
    return Response(csv, media_type="text/csv", headers={"Content-Disposition": f'attachment; filename="{fname}"'})

# ---------------- free-text tagging ----------------
from pydantic import BaseModel
TOKEN_RE = re.compile(r"[^\W_]+|[^\w\s]|_")          # NCBI-style: words, and every punctuation mark on its own
SENT_RE  = re.compile(r"(?<=[.!?])\s+(?=[A-Z(\[])")

def split_text(text):
    sents = []
    for line in text.splitlines():
        for s in SENT_RE.split(line.strip()):
            toks = TOKEN_RE.findall(s)
            if toks: sents.append(toks)
    return sents

class TagRequest(BaseModel):
    text: str
    model: str

@app.post("/tag")
def tag_text(req: TagRequest):
    if req.model not in MODEL_NAMES: raise HTTPException(400, f"Unknown model '{req.model}'. Use smollm or sapbert.")
    if len(req.text) > 5000: raise HTTPException(400, "Text is too long. Keep it under 5,000 characters.")
    sents = split_text(req.text)
    if not sents: raise HTTPException(400, "Type some text to tag.")
    if len(sents) > 40: raise HTTPException(400, f"That's {len(sents)} sentences. Tag up to 40 at a time, or upload a CSV.")
    if not GPU_LOCK.acquire(timeout=15):
        raise HTTPException(409, "The GPU is busy with a file run. Try again when it finishes.")
    try:
        t0 = time.time()
        base = PREDICTORS[req.model](sents)
        scores = head_span_scores(sents, base)
    finally:
        GPU_LOCK.release()
    thr = DROP_THR[req.model]
    out = [{"tokens": w, "base": b, "head": veto(b, s, thr),
            "scores": [{"start": a, "end": e, "score": round(float(x), 3)} for (a, e), x in s.items()]}
           for w, b, s in zip(sents, base, scores)]
    return {"model": req.model, "model_name": MODEL_NAMES[req.model], "threshold": thr,
            "seconds": round(time.time() - t0, 2), "sentences": out}

load_ncbi()
print(f"API ready | NCBI test: {len(DATASETS['ncbi']['tokens'])} sentences")

## Start the server

In [ ]:
import uvicorn

PORT = 8000
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for _ in range(60):
    try:
        if requests.get(f"http://localhost:{PORT}/health", timeout=2).ok: break
    except Exception: time.sleep(0.5)
else:
    raise RuntimeError("Server did not start")

PUBLIC_URL = None
if TUNNEL == "ngrok":
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    PUBLIC_URL = ngrok.connect(PORT, "http").public_url
else:
    if not os.path.exists("cloudflared"):
        subprocess.run(["wget", "-q", "-O", "cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod("cloudflared", 0o755)
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    t0 = time.time()
    while time.time() - t0 < 60 and PUBLIC_URL is None:
        line = tunnel.stdout.readline()
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
        if m: PUBLIC_URL = m.group(0)
    if PUBLIC_URL is None:
        raise RuntimeError("Cloudflare tunnel did not start. Set TUNNEL = 'ngrok' in the settings cell and run again.")
    threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()   # keep the pipe drained
    time.sleep(3)   # give the new hostname a moment to resolve

print("\n" + "=" * 70)
print("Backend URL (paste this into the web app):\n\n   ", PUBLIC_URL)
if FRONTEND_URL:
    print("\nOr open the app already connected:\n\n   ", f"{FRONTEND_URL.rstrip('/')}/?api={PUBLIC_URL}")
print("=" * 70)

## Live job log

In [ ]:
# Leave this cell running during the demo. It shows each job as it happens.
# Stop it any time with the stop button - the server keeps running in the background.
seen = set()
try:
    while True:
        for jid, j in list(JOBS.items()):
            tag = (jid, j["status"])
            if tag not in seen:
                seen.add(tag)
                print(time.strftime("%H:%M:%S"), f"job {jid} | {MODEL_NAMES[j['model']]} | {DATASETS[j['dataset']]['name']} | {j['status']}"
                      + (f" | {j['error']}" if j["error"] else ""))
        time.sleep(2)
except KeyboardInterrupt:
    print("Monitor stopped. Server is still running at", PUBLIC_URL)